# Floating-Point Representation
### Interactive Teaching Notebook

**Reference:** Solomon, *Numerical Algorithms*, Ch. 1

---

Computers have **finite storage**, so they cannot represent every real number exactly.  
This notebook walks through the key consequences of that limitation.

**Contents**
1. Why 1/3 cannot be stored exactly in binary
2. The danger of `==` on floats
3. The fix: epsilon-tolerance comparison
4. Machine epsilon — the fundamental unit of rounding error
5. IEEE 754 double precision — what's inside a `float64`
6. Representable numbers are NOT uniformly spaced
7. Catastrophic cancellation
8. Choosing a tolerance — the design trade-off
9. Summary

In [ ]:
import sys, struct, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({'figure.dpi': 110, 'font.size': 12})

---
## 1 · Why 1/3 Cannot Be Stored Exactly in Binary

Computers store numbers in **binary (base-2)**. Some fractions that look simple in base-10 become **infinite** in base-2:

| Fraction | Base-10 | Base-2 |
|----------|---------|--------|
| 1/2 | 0.5 *(finite)* | 0.1 *(finite)* |
| 1/4 | 0.25 *(finite)* | 0.01 *(finite)* |
| 1/3 | 0.333… *(infinite)* | 0.010101… *(infinite)* |
| 1/10 | 0.1 *(finite)* | 0.000110011… *(infinite)* |

Because memory is **finite**, the computer must **round** to the nearest representable value — introducing a tiny but real error.

In [ ]:
x = 1.0
y = x / 3.0

print(f"x        = {x}")
print(f"y = x/3  = {y:.20f}")
print(f"           true 1/3 = 0.33333333333333333333...")
print()
print(f"y * 3    = {y * 3.0:.20f}")
print(f"           (should be exactly 1.0)")
print()
print(f"Error: x - y*3 = {x - y*3.0:.4e}")

---
## 2 · The Danger of `==` on Floats

The following C++ code prints **`"They are NOT equal."`** — counter-intuitively:

```cpp
double x = 1.0;
double y = x / 3.0;
if (x == y * 3.0)
    cout << "They are equal!";
else
    cout << "They are NOT equal.";
```

The Python equivalent:

In [ ]:
x = 1.0
y = x / 3.0

if x == y * 3.0:
    print('Result: "They are equal!"')
else:
    print('Result: "They are NOT equal."  <-- counter-intuitive!')

print()
print(f"x     = {x!r}")
print(f"y*3   = {y*3.0!r}")
print(f"diff  = {x - y*3.0}")

> **Rule:** Rarely (if ever) use `==` to compare floating-point numbers that were computed via arithmetic.  
> Use a **tolerance** instead.

---
## 3 · The Fix: Epsilon-Tolerance Comparison

Instead of asking *"are these exactly equal?"* we ask *"are they close enough?"*

The C++ fix from the notes:
```cpp
if (fabs(x - y*3.0) < numeric_limits<double>::epsilon)
    cout << "They are equal!";
```

Python equivalents:

In [ ]:
eps_machine = sys.float_info.epsilon
print(f"Machine epsilon = {eps_machine:.4e}")
print(f"  (smallest e such that 1.0 + e != 1.0 in float64)")
print()

diff = abs(x - y * 3.0)
print(f"|x - y*3| = {diff:.4e}")
print()

# Absolute tolerance (C++ style)
if diff < eps_machine:
    print("abs tolerance:  They are equal!  [OK]")
else:
    print("abs tolerance:  NOT equal -- trying relative...")

# Relative tolerance (more robust for large numbers)
rtol = 1e-9
if diff <= rtol * max(abs(x), abs(y * 3.0)):
    print(f"rel tolerance (1e-9):  They are equal!  [OK]")

# Best practice: math.isclose
print()
print(f"math.isclose(x, y*3) = {math.isclose(x, y * 3.0)}")
print("  (checks: |a-b| <= max(rel_tol * max(|a|,|b|), abs_tol))")

> **Best practice (Python ≥ 3.5):** use `math.isclose(a, b, rel_tol=1e-9, abs_tol=0.0)`

---
## 4 · Machine Epsilon — The Fundamental Unit of Rounding Error

**Machine epsilon** ($\varepsilon_{\text{mach}}$) is the smallest positive number such that:

$$1.0 + \varepsilon_{\text{mach}} \neq 1.0 \quad \text{(in floating-point arithmetic)}$$

For IEEE 754 double precision: $\varepsilon_{\text{mach}} = 2^{-52} \approx 2.22 \times 10^{-16}$

It also equals the **maximum relative rounding error** per arithmetic operation.

In [ ]:
# Find machine epsilon empirically
e = 1.0
steps = 0
while 1.0 + e != 1.0:
    e /= 2.0
    steps += 1
e *= 2.0  # last value that still mattered

print(f"Empirical eps_mach ({steps} halvings): {e:.6e}")
print(f"sys.float_info.epsilon:                {sys.float_info.epsilon:.6e}")
print(f"Formula 2^(-52):                       {2**-52:.6e}")
print()

half_eps = sys.float_info.epsilon / 2
print("Boundary test:")
print(f"  1.0 + eps/2 == 1.0 ?  {1.0 + half_eps == 1.0}   (rounded away -- invisible)")
print(f"  1.0 + eps   == 1.0 ?  {1.0 + sys.float_info.epsilon == 1.0}  (just distinguishable)")

In [ ]:
# Visualise the boundary
deltas = np.logspace(-17, -14, 500)
differs = np.array([1.0 + d != 1.0 for d in deltas], dtype=float)

fig, ax = plt.subplots(figsize=(9, 2.5))
ax.semilogx(deltas, differs, color='steelblue', lw=2)
ax.axvline(sys.float_info.epsilon, color='tomato', lw=1.5, ls='--',
           label=f'eps_mach = {sys.float_info.epsilon:.2e}')
ax.set_xlabel('delta  (added to 1.0)')
ax.set_ylabel('1.0 + delta != 1.0')
ax.set_yticks([0, 1]); ax.set_yticklabels(['False', 'True'])
ax.set_title('When does 1.0 + delta become distinguishable from 1.0?')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5 · IEEE 754 Double Precision — What's Inside a `float64`

A Python `float` (C `double`) occupies **64 bits**:

```
+------+-------------------+------------------------------------------+
| sign |    exponent       |              mantissa                    |
|  1b  |      11b          |                52b                       |
+------+-------------------+------------------------------------------+

value = (-1)^sign  *  2^(exponent - 1023)  *  (1 + mantissa / 2^52)
```

The 52-bit mantissa gives **~15–17 significant decimal digits**.

In [ ]:
def decode_float64(f):
    bits = struct.unpack('Q', struct.pack('d', f))[0]
    sign     = (bits >> 63) & 0x1
    exponent = (bits >> 52) & 0x7FF
    mantissa =  bits        & 0x000FFFFFFFFFFFFF
    bits_str = f"{bits:064b}"
    return sign, exponent, mantissa, bits_str

values = [1.0, 0.5, 1.0/3.0, -2.75, 0.1]
print(f"{'Value':>10}  {'Sign':>4}  {'Exp (biased)':>16}  {'First 20 mantissa bits'}")
print("-" * 70)
for v in values:
    s, e, m, b = decode_float64(v)
    print(f"{v:>10.6f}  {b[0]:>4}  {b[1:12]:>16} ({e:4d})  {b[12:32]}...")

In [ ]:
# Visualise the 64 bits of a number
def plot_bits(value, ax):
    s, e, m, b = decode_float64(value)
    colors = (['#e74c3c'] +            # sign (1 bit, red)
              ['#3498db'] * 11 +       # exponent (11 bits, blue)
              ['#2ecc71'] * 52)        # mantissa (52 bits, green)
    for i, (bit, col) in enumerate(zip(b, colors)):
        ax.add_patch(mpatches.FancyBboxPatch(
            (i, 0), 0.9, 0.9,
            boxstyle='round,pad=0.05',
            fc=col if bit == '1' else '#ecf0f1',
            ec='#95a5a6', lw=0.4))
        if i in (0, 1, 12):
            ax.text(i + 0.45, -0.35, ['S', 'E', 'M'][sum([i>=0, i>=1, i>=12])-1],
                    ha='center', fontsize=7, color='#7f8c8d')
    ax.set_xlim(-0.5, 64.5)
    ax.set_ylim(-0.6, 1.1)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f"{value}  (sign={s}, exp={e}, first mantissa bits: {b[12:20]}...)",
                 fontsize=10)

fig, axes = plt.subplots(4, 1, figsize=(13, 5))
for ax, v in zip(axes, [1.0, 1.0/3.0, 0.1, -2.75]):
    plot_bits(v, ax)

# Legend
legend_patches = [
    mpatches.Patch(color='#e74c3c', label='Sign (1b)'),
    mpatches.Patch(color='#3498db', label='Exponent (11b)'),
    mpatches.Patch(color='#2ecc71', label='Mantissa (52b)'),
    mpatches.Patch(color='#ecf0f1', ec='#95a5a6', label='bit = 0'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=9)
fig.suptitle('IEEE 754 bit layout for selected float64 values', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

Notice how **1/3** and **0.1** have repeating mantissa patterns — their binary representations are infinite, so the 52-bit mantissa truncates them.

---
## 6 · Representable Numbers Are NOT Uniformly Spaced

The gap between adjacent `float64` values **scales with magnitude** (logarithmic spacing).

$$\text{gap}(x) \approx \varepsilon_{\text{mach}} \cdot |x|$$

This is captured by the **ULP** (Unit in the Last Place).

In [ ]:
magnitudes = [1e0, 1e1, 1e3, 1e6, 1e9, 1e12, 1e15]
print(f"{'x':>10}   {'Gap to next float64':>22}   {'# gaps in [x, 2x]':>20}")
print("-" * 60)
for xv in magnitudes:
    gap = math.ulp(xv)
    n_gaps = int(xv / gap)
    print(f"{xv:>10.0e}   {gap:>22.3e}   {n_gaps:>20,}")

In [ ]:
# Visualise representable floats near 1.0 and near 100.0 (zoomed)
fig, axes = plt.subplots(1, 2, figsize=(12, 2.5))

for ax, centre, half_range in zip(axes, [1.0, 100.0], [5e-15, 5e-13]):
    # generate representable floats in range
    pts = []
    v = centre - half_range
    while v <= centre + half_range:
        pts.append(v)
        v = math.nextafter(v, math.inf)
        if len(pts) > 300:
            break

    ax.vlines(pts, 0, 1, color='steelblue', lw=1.2)
    ax.axvline(centre, color='tomato', lw=1.5, ls='--', label=f'x = {centre}')
    ax.set_yticks([])
    ax.set_xlabel(f'Value (zoomed near {centre})')
    ax.set_title(f'Representable floats near {centre}  |  gap = {math.ulp(centre):.2e}')
    ax.legend(fontsize=9)

fig.suptitle('Tick marks = representable float64 values (not uniformly spaced!)', fontsize=11)
plt.tight_layout()
plt.show()

> **Consequence:** An absolute tolerance of `1e-10` is very generous near 1.0 but is **smaller than one ULP** near `1e12`.  
> Always prefer **relative tolerances**: $|a - b| / \max(|a|, |b|) < \text{rel\_tol}$

---
## 7 · Catastrophic Cancellation

Subtracting two **nearly-equal** numbers can destroy almost all significant digits — even if each operand was accurate to full machine precision.

In [ ]:
a = 1.000000000000001
b = 1.000000000000000

result = a - b
exact  = 1e-15
rel_err = abs(result - exact) / abs(exact)

print(f"a              = {a:.16f}")
print(f"b              = {b:.16f}")
print(f"a - b          = {result:.6e}   (computed)")
print(f"true value     = {exact:.6e}")
print(f"relative error = {rel_err:.1%}")
print()
print("Both a and b are accurate, but their difference lost most digits!")

In [ ]:
# Classic example: quadratic formula
# x^2 + b*x + 1 = 0  with large b => b^2 >> 4ac => cancellation

def roots_naive(b_coef, a_coef=1.0, c_coef=1.0):
    disc = b_coef**2 - 4*a_coef*c_coef
    return (-b_coef + math.sqrt(disc)) / (2*a_coef)

def roots_stable(b_coef, a_coef=1.0, c_coef=1.0):
    disc = b_coef**2 - 4*a_coef*c_coef
    # rationalise the numerator to avoid cancellation
    return (-2*c_coef) / (b_coef + math.sqrt(disc))

print(f"{'b':>10}  {'Naive':>18}  {'Stable':>18}  {'Exact (-1/b)':>18}  {'Naive rel err':>14}")
print("-" * 85)
for b_val in [1e2, 1e4, 1e6, 1e8, 1e10, 1e12]:
    naive  = roots_naive(b_val)
    stable = roots_stable(b_val)
    exact  = -1.0 / b_val
    err    = abs(naive - exact) / abs(exact) if exact != 0 else float('nan')
    print(f"{b_val:>10.0e}  {naive:>18.10e}  {stable:>18.10e}  {exact:>18.10e}  {err:>14.2%}")

In [ ]:
# Plot relative error vs b coefficient
b_vals = np.logspace(1, 15, 200)
errs_naive  = []
errs_stable = []

for bv in b_vals:
    exact = -1.0 / bv
    try:
        en = abs(roots_naive(bv)  - exact) / abs(exact)
        es = abs(roots_stable(bv) - exact) / abs(exact)
    except Exception:
        en = es = float('nan')
    errs_naive.append(en)
    errs_stable.append(es)

fig, ax = plt.subplots(figsize=(9, 4))
ax.loglog(b_vals, errs_naive,  label='Naive formula', color='tomato',    lw=2)
ax.loglog(b_vals, errs_stable, label='Stable formula', color='steelblue', lw=2)
ax.axhline(sys.float_info.epsilon, ls='--', color='grey', lw=1,
           label=f'Machine epsilon ({sys.float_info.epsilon:.1e})')
ax.set_xlabel('b coefficient')
ax.set_ylabel('Relative error in smaller root')
ax.set_title('Catastrophic Cancellation in Quadratic Formula')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

The naive formula catastrophically loses precision as $b$ grows, while the stable (rationalised) formula stays near machine precision.

---
## 8 · Choosing a Tolerance — The Design Trade-Off

The tolerance $\varepsilon$ defines the line between *"equal"* and *"different"*. Choosing it is an **application-specific design decision**.

| | Tolerance too **small** | Tolerance too **large** |
|---|---|---|
| **Effect** | Equal values treated as different (false negatives) | Different values treated as equal (false positives) |
| **Symptom** | Algorithm loops forever; wrong branch taken | Wrong results accepted silently |

### Practical guidelines

| Use case | Suggested tolerance |
|----------|--------------------|
| General float comparison | `rel_tol = 1e-9` |
| After $n$ floating-point ops | `rel_tol = n * eps_mach` |
| Physical simulation (mm → km scale) | `abs_tol` matched to domain units |
| Iterative solver convergence | `rel_tol` relative to iterate size |
| Unit tests / CI assertions | `math.isclose()` or `np.testing.assert_allclose()` |

In [ ]:
# Interactive tolerance demo
a_val = 1.0
b_val = 1.0 + 1e-10   # slightly different

for tol in [1e-16, 1e-12, 1e-10, 1e-9, 1e-6, 1e-3]:
    equal = math.isclose(a_val, b_val, rel_tol=tol)
    tag = "  <- boundary!" if not math.isclose(a_val, b_val, rel_tol=tol*10) else ""
    print(f"  rel_tol = {tol:.0e}  -->  isclose = {equal}{tag}")

---
## 9 · Summary

| # | Key Point |
|---|----------|
| 1 | Computers use **finite binary representations** — most fractions (including 1/3 and 0.1) cannot be stored exactly |
| 2 | **Never use `==`** to compare floats computed via arithmetic — use `math.isclose()` |
| 3 | **Machine epsilon** $\varepsilon_{\text{mach}} = 2^{-52} \approx 2.22 \times 10^{-16}$ is the maximum relative rounding error per operation |
| 4 | Representable numbers are **logarithmically spaced** — use relative tolerances for large values |
| 5 | **Catastrophic cancellation** destroys digits when subtracting nearly-equal numbers — reformulate when possible |
| 6 | The tolerance is an **application-specific design choice** — too small causes false mismatches, too large hides errors |

---

*Reference: Solomon, Numerical Algorithms, Ch. 1 (Floating Point)*